In [34]:
from pathlib import Path
import pandas as pd

# ============================================================
# Paths
# ============================================================

AQI_FOLDER = Path("../data/raw/aqi")
OUTPUT = Path("../data/processed/merged_aqi.csv")

# ============================================================
# Get all AQI Excel files
# ============================================================

files = sorted(
    f for f in AQI_FOLDER.glob("*.xlsx")
    if not f.name.startswith("~$")
)

if not files:
    raise FileNotFoundError(f"No Excel files found in: {AQI_FOLDER}")

dataframes = []

print("=" * 70)
print("MERGING AQI DATASETS (8 PM DAILY READINGS)")
print("=" * 70)

# ============================================================
# Process each file
# ============================================================

for file in files:

    print(f"Processing: {file.name}")

    try:

        # --------------------------------------------------------
        # Read metadata
        # --------------------------------------------------------

        metadata = pd.read_excel(file, header=None)

        station_name = str(metadata.iloc[6, 1]).split(",")[0].strip()

        # --------------------------------------------------------
        # Read AQI data
        # --------------------------------------------------------

        df = pd.read_excel(file, header=16)

        # Remove completely empty rows & columns
        df.dropna(how="all", inplace=True)
        df.dropna(axis=1, how="all", inplace=True)

        # Clean column names
        df.columns = df.columns.astype(str).str.strip()

        # --------------------------------------------------------
        # Convert dates (IMPORTANT: DD-MM-YYYY)
        # --------------------------------------------------------

        df["From Date"] = pd.to_datetime(
            df["From Date"],
            format="%d-%m-%Y %H:%M",
            errors="raise"
        )

        df["To Date"] = pd.to_datetime(
            df["To Date"],
            format="%d-%m-%Y %H:%M",
            errors="raise"
        )

        # --------------------------------------------------------
        # Keep only 8 PM observations
        # --------------------------------------------------------

        df = df[df["From Date"].dt.hour == 20].copy()

        # --------------------------------------------------------
        # Keep one date column
        # --------------------------------------------------------

        df.rename(
            columns={"From Date": "Date"},
            inplace=True
        )

        df["Date"] = df["Date"].dt.date

        # Remove To Date
        if "To Date" in df.columns:
            df.drop(columns=["To Date"], inplace=True)

        # --------------------------------------------------------
        # Add metadata
        # --------------------------------------------------------

        df["Station"] = station_name
        df["Source_File"] = file.stem

        dataframes.append(df)

        print(f"Rows kept : {len(df)}")

    except Exception as e:
        print(f"Error processing {file.name}")
        print(e)

# ============================================================
# Merge all stations
# ============================================================

merged = pd.concat(
    dataframes,
    ignore_index=True
)

# ============================================================
# Remove duplicate Station-Date records
# ============================================================

merged.drop_duplicates(
    subset=["Station", "Date"],
    inplace=True
)

# ============================================================
# Sort dataset
# ============================================================

merged.sort_values(
    by=["Station", "Date"],
    inplace=True,
    ignore_index=True
)

# ============================================================
# Save merged dataset
# ============================================================

OUTPUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

merged.to_csv(
    OUTPUT,
    index=False
)

# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 70)
print("AQI FILES MERGED SUCCESSFULLY")
print("=" * 70)

print(f"Files Processed : {len(files)}")
print(f"Stations        : {merged['Station'].nunique()}")
print(f"Rows            : {merged.shape[0]}")
print(f"Columns         : {merged.shape[1]}")

print("\nRows per Station")
print(merged.groupby("Station").size())

print("\nMissing Values")
print(merged.isnull().sum())

print("\nColumns")
for col in merged.columns:
    print(f" - {col}")

print(f"\nSaved to : {OUTPUT}")

print("=" * 70)

MERGING AQI DATASETS (8 PM DAILY READINGS)
Processing: AIRPORT 24-26.xlsx
Rows kept : 881
Processing: AMBERNATH 24-26.xlsx
Rows kept : 881
Processing: ANDHERI 24-26.xlsx
Rows kept : 881
Processing: BADLAPUR 24-26.xlsx
Rows kept : 881
Processing: BELAPUR 24-26.xlsx
Rows kept : 881
Processing: BHANDUP 24-26.xlsx
Rows kept : 881
Processing: BHAYANDAR 24-26.xlsx
Rows kept : 881
Processing: BHIWANDI 24-26.xlsx
Rows kept : 881
Processing: BKC 24-26.xlsx
Rows kept : 881
Processing: BOISAR 24-26.xlsx
Rows kept : 881
Processing: BORIVALI 24-26.xlsx
Rows kept : 881
Processing: BYCULLA 24-26.xlsx
Rows kept : 881
Processing: CHEMBUR 24-26.xlsx
Rows kept : 881
Processing: COLABA 24-26.xlsx
Rows kept : 881
Processing: DEONAR 24-26.xlsx
Rows kept : 881
Processing: GHATKOPAR 24-26.xlsx
Rows kept : 881
Processing: KALUNAGAR D 24-26.xlsx
Rows kept : 881
Processing: KANDIVALI 24-26.xlsx
Rows kept : 881
Processing: KASARVADAVALI 24-26.xlsx
Rows kept : 881
Processing: KHADAKPADA KALYAN 24-26.xlsx
Rows kept